# V24.3 full-199 score validation

Frozen two-shard population validation of the V24.3 short-fragment shadow. This notebook performs inference and official evaluation only. It does not train, tune thresholds, create hybrids, mutate production graphs, or generate submissions.

Run this notebook twice: first with `SHARD_INDEX = 0`, then with `SHARD_INDEX = 1`. Attach the Biohub competition data, `pilkwang/biohub-tracking-support-pack-50ep-v1`, and the private frozen E016 checkpoint dataset. Enable a Kaggle GPU and Internet.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time

BRANCH = "v24-score-first-tracking"
EXPECTED_COMMIT = "7ec286f62ce08803bc7117d9d1e9e80d25bbf3cd"
ROOT = Path("/tmp/Atabey")

if not ROOT.exists():
    subprocess.run(
        [
            "git" ,
            "clone" ,
            "--branch" ,
            BRANCH,
            "https://github.com/drosadocastro-bit/Atabey.git" ,
            str(ROOT),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)

subprocess.run(
    ["git", "-C", str(ROOT), "checkout", "--detach", EXPECTED_COMMIT],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()
assert actual_commit == EXPECTED_COMMIT, (actual_commit, EXPECTED_COMMIT)

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = f"{ROOT}:{ROOT / 'src'}:{ROOT / 'scripts'}"
print("Atabey commit verified:", actual_commit)

In [ ]:
pinned_official_packages = [
    "git+https://github.com/royerlab/tracksdata.git@39dccf3a243e44274759468cb31b2ad9e7fc1d09" ,
    "git+https://github.com/royerlab/kaggle-cell-tracking-competition.git@075fc5f5a52d11077f9dc2b074644618f26939e2" ,
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pinned_official_packages],
    check=True,
)

official_runtime_packages = [
    "bidict>=0.23.1", "blosc2", "dask", "geff>=1.1.3.1.1",
    "ilpy>=0.5.1", "imagecodecs", "numba", "numcodecs>=0.13",
    "numpy>2", "polars>=1.36.0", "psygnal>=0.14.0", "pyarrow",
    "rich", "rustworkx>=0.17.1", "scikit-image>=0.24.0",
    "sqlalchemy>=2", "tqdm", "typing-extensions", "zarr>=3.0.10",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *official_runtime_packages],
    check=True,
)

import numpy as np
import scipy
import torch

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before full-199 validation"

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
EXPECTED_CHECKPOINT_SHA256 = (
    "02e1d65756c3dc5928f68a66a8b0ef99be2a6905fa7bc017aa1d87dbe632fd03"
)

train_candidates = []
for pattern in ("*/train", "*/*/train", "*/*/*/train"):
    for candidate in INPUT_ROOT.glob(pattern):
        if (
            candidate.is_dir()
            and len(list(candidate.glob("*.zarr"))) == 199
            and len(list(candidate.glob("*.geff"))) == 199
        ):
            train_candidates.append(candidate)
train_candidates = sorted(set(train_candidates))
assert len(train_candidates) == 1, f"Expected one 199-sample train directory: {train_candidates}"
TRAIN_DIR = train_candidates[0]

auxiliary_roots = sorted(
    root
    for root in INPUT_ROOT.iterdir()
    if root.is_dir() and not TRAIN_DIR.is_relative_to(root)
)
assert auxiliary_roots, "No auxiliary Kaggle input roots found"

predictor_candidates = []
for root in auxiliary_roots:
    predictor_candidates.extend(root.rglob("predict_unet_transformer.py"))
predictor_candidates = sorted(
    path for path in set(predictor_candidates) if path.parent.name == "scripts"
)
support_candidates = sorted({path.parents[1] for path in predictor_candidates})
assert len(support_candidates) == 1, f"Expected one support repository: {support_candidates}"
SUPPORT_REPO = support_candidates[0]

weight_candidates = []
for root in auxiliary_roots:
    weight_candidates.extend(root.rglob("edge_predictor_best.pth"))
matching_weights = [
    candidate
    for candidate in sorted(set(weight_candidates))
    if (candidate.parent / "config.json").exists()
    and hashlib.sha256(candidate.read_bytes()).hexdigest() == EXPECTED_CHECKPOINT_SHA256
]
assert len(matching_weights) == 1, f"Expected one exact frozen checkpoint: {matching_weights}"
WEIGHTS = matching_weights[0]

checkpoint_config = json.loads((WEIGHTS.parent / "config.json").read_text(encoding="utf-8"))
assert checkpoint_config["window_size"] == 2
assert checkpoint_config["downsample"] == [1, 4, 4]
assert checkpoint_config["unet_out_channels"] == 32
assert checkpoint_config["pool_kernel_um"] == 5.0

print("Train:", TRAIN_DIR)
print("Support:", SUPPORT_REPO)
print("Checkpoint:", WEIGHTS)

In [ ]:
focused_tests = [
    ROOT / "tests/test_v24_3_full_199_score_validation.py" ,
    ROOT / "tests/test_v24_score_first_tracking_preregistration.py" ,
    ROOT / "tests/test_unet_graph.py" ,
    ROOT / "tests/test_official_tracking_metric.py" ,
]
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", *map(str, focused_tests)],
    check=True,
    env=RUN_ENV,
)
print("Full-199 contract, graph conversion, and official metric tests passed")

## Execution gate

Set `SHARD_INDEX` to 0 for the first run and 1 for the second. Do not change any other execution setting between runs. A shard is not a full-199 result; both artifacts must pass strict merge validation.

In [ ]:
SHARD_INDEX = 0  # Run 0 first; then change only this value to 1.
SHARD_COUNT = 2
AUTHORIZE_FULL_199_SCORE_VALIDATION = True

assert SHARD_INDEX in {0, 1}
assert SHARD_COUNT == 2
assert AUTHORIZE_FULL_199_SCORE_VALIDATION is True

CONTRACT_PATH = ROOT / "tests/fixtures/v24_3_full_199_score_validation.json"
AUTHORIZATION_REPORT = ROOT / "v24_3_short_fragment_shadow_full_27_report.json"
contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
authorization = json.loads(AUTHORIZATION_REPORT.read_text(encoding="utf-8"))
assert contract["population"]["expected_samples"] == 199
assert contract["population"]["shard_count"] == SHARD_COUNT
assert authorization["decision"] == "GO_TO_FULL_199_SCORE_VALIDATION"
assert authorization["authorization"]["full_199_score_validation"] is True
assert contract["boundaries"]["submission_authorized"] is False
assert contract["boundaries"]["production_graph_mutation"] is False

OUTPUT_DIR = Path(f"/kaggle/working/v24_3_full_199_shard_{SHARD_INDEX}")
print("Shard:", SHARD_INDEX, "of", SHARD_COUNT)
print("Output:", OUTPUT_DIR)

In [ ]:
command = [
    sys.executable,
    "-u" ,
    str(ROOT / "scripts/run_v24_3_full_199_score_validation.py"),
    "--train-dir", str(TRAIN_DIR),
    "--support-repo", str(SUPPORT_REPO),
    "--weights", str(WEIGHTS),
    "--contract", str(CONTRACT_PATH),
    "--authorization-report", str(AUTHORIZATION_REPORT),
    "--output-dir", str(OUTPUT_DIR),
    "--shard-index", str(SHARD_INDEX),
    "--shard-count", str(SHARD_COUNT),
    "--unet-batch-size", "4" ,
    "--resume" ,
    "--verify-determinism" ,
]
started = time.time()
subprocess.run(command, check=True, env=RUN_ENV)
elapsed_seconds = time.time() - started
print("Shard elapsed hours:", elapsed_seconds / 3600.0)

In [ ]:
summary = json.loads((OUTPUT_DIR / "summary.json").read_text(encoding="utf-8"))
expected_count = 100 if SHARD_INDEX == 0 else 99
assert summary["decision"] == "FULL_199_SCORE_VALIDATION_SHARD_COMPLETE"
assert summary["sample_count"] == expected_count
assert summary["expected_sample_count"] == expected_count
assert summary["complete_shard"] is True
assert summary["determinism_verified"] is True
assert summary["assignment_enabled"] is False
assert summary["hybrid_enabled"] is False
assert summary["production_graph_mutation"] is False
assert summary["submission_authorized"] is False
assert len(list((OUTPUT_DIR / "samples").glob("*.json"))) == expected_count

print("Decision:", summary["decision"])
print("Samples:", summary["sample_count"])
print("Checkpoint usage:", summary["checkpoint_usage_counts"])
print("V24.3 vs V19:", summary["v24_3_vs_v19"])

In [ ]:
BUNDLE = Path(f"/kaggle/working/v24_3_full_199_shard_{SHARD_INDEX}_outputs")
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
shutil.copytree(OUTPUT_DIR, BUNDLE / "run")
shutil.copy2(CONTRACT_PATH, BUNDLE / CONTRACT_PATH.name)
shutil.copy2(AUTHORIZATION_REPORT, BUNDLE / AUTHORIZATION_REPORT.name)

run_record = {
    "mode": "full_199_score_validation_shard" ,
    "shard_index": SHARD_INDEX,
    "shard_count": SHARD_COUNT,
    "atabey_commit": EXPECTED_COMMIT,
    "checkpoint_sha256": EXPECTED_CHECKPOINT_SHA256,
    "support_predictor_sha256": summary["provenance"]["predictor_sha256"],
    "elapsed_seconds": elapsed_seconds,
    "decision": summary["decision"],
    "no_training": True,
    "submission_authorized": False,
}
(BUNDLE / "notebook_run_record.json").write_text(
    json.dumps(run_record, indent=2, sort_keys=True) + "\n" ,
    encoding="utf-8" ,
)
archive = shutil.make_archive(str(BUNDLE), "zip", BUNDLE)
print("Download:", archive)

## Interpretation boundary

Download the shard archive before starting the other shard. The two archives must be merged with `scripts/merge_v24_3_full_199_score_validation.py`; neither shard alone is a population result. The 172 checkpoint-training samples provide population context, not independent generalization evidence. This validation does not authorize submission or production graph mutation.